In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import json
import re
import random
import time
from urllib.parse import urljoin
import warnings

warnings.filterwarnings("ignore")

In [9]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/138.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9"
}

BASE_URL = "https://www.carwale.com"

In [11]:
def get_soup(url):

    response = requests.get(url, headers=HEADERS)

    if response.status_code == 200:
        return BeautifulSoup(response.text, "html.parser")

    print("Failed :", url)
    return None

In [13]:
def get_brand_urls():

    soup = get_soup(BASE_URL + "/new-cars/")

    brand_urls = set()

    for tag in soup.find_all("a", href=True):

        href = tag["href"]

        if not re.fullmatch(r"/[a-z0-9-]+-cars/", href):
            continue

        brand_urls.add(BASE_URL + href)

    return sorted(brand_urls)

In [15]:
brand_urls = get_brand_urls()

print("Total Brands :", len(brand_urls))

Total Brands : 38


In [17]:
def get_car_urls(brand_url):

    soup = get_soup(brand_url)

    car_urls = set()

    brand_slug = brand_url.rstrip("/").split("/")[-1]

    bad_words = [
        "price-in",
        "expert-reviews",
        "images",
        "videos",
        "news",
        "old-generation",
        "prime",
        "jtp",
        "genx",
        "2011","2012","2013","2014","2015",
        "2016","2017","2018","2019",
        "2020","2021","2022","2023","2024",
        "1991"
    ]

    for tag in soup.find_all("a", href=True):

        href = tag["href"]

        if not href.startswith(f"/{brand_slug}/"):
            continue

        if href.count("/") != 3:
            continue

        if "#" in href or "?" in href:
            continue

        if any(word in href.lower() for word in bad_words):
            continue

        car_urls.add(BASE_URL + href)

    return sorted(car_urls)

In [19]:
all_car_urls = []

for brand in brand_urls:

    print(f"Scraping : {brand}")

    try:
        urls = get_car_urls(brand)
        all_car_urls.extend(urls)

    except Exception as e:
        print(e)

    time.sleep(random.uniform(1,2))

all_car_urls = sorted(set(all_car_urls))

print("Total Cars :",len(all_car_urls))

Scraping : https://www.carwale.com/aston-martin-cars/
Scraping : https://www.carwale.com/audi-cars/
Scraping : https://www.carwale.com/bentley-cars/
Scraping : https://www.carwale.com/bmw-cars/
Scraping : https://www.carwale.com/byd-cars/
Scraping : https://www.carwale.com/citroen-cars/
Scraping : https://www.carwale.com/compare-cars/
Scraping : https://www.carwale.com/ferrari-cars/
Scraping : https://www.carwale.com/force-motors-cars/
Scraping : https://www.carwale.com/honda-cars/
Scraping : https://www.carwale.com/hyundai-cars/
Scraping : https://www.carwale.com/isuzu-cars/
Scraping : https://www.carwale.com/jaguar-cars/
Scraping : https://www.carwale.com/jeep-cars/
Scraping : https://www.carwale.com/kia-cars/
Scraping : https://www.carwale.com/lamborghini-cars/
Scraping : https://www.carwale.com/land-rover-cars/
Scraping : https://www.carwale.com/lexus-cars/
Scraping : https://www.carwale.com/lotus-cars/
Scraping : https://www.carwale.com/mahindra-cars/
Scraping : https://www.carwal

In [21]:
def scrape_car(url):

    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    ld_json = None

    for script in soup.find_all("script", type="application/ld+json"):

        try:
            data = json.loads(script.string)

            if "@graph" in data:

                for item in data["@graph"]:

                    if item.get("@type") == "Car":
                        ld_json = item
                        break

        except:
            continue

    if ld_json is None:
        return None

    # -----------------------
    # Price
    # -----------------------

    price = np.nan

    if "offers" in ld_json:

        offer = ld_json["offers"]

        if isinstance(offer, dict):
            price = offer.get("price", np.nan)

    # -----------------------
    # Brand
    # -----------------------

    brand = np.nan

    if "brand" in ld_json:

        if isinstance(ld_json["brand"], dict):
            brand = ld_json["brand"].get("name", np.nan)

        else:
            brand = ld_json["brand"]

    # -----------------------
    # Description
    # -----------------------

    description = ld_json.get("description", "")

    fuel = np.nan
    transmission = np.nan
    engine = np.nan
    mileage = np.nan
    seating = np.nan

    fuel_match = re.search(
        r"(Petrol|Diesel|CNG|Electric|Hybrid)",
        description,
        re.I
    )

    if fuel_match:
        fuel = fuel_match.group(1).title()

    transmission_match = re.search(
        r"(Manual|Automatic)",
        description,
        re.I
    )

    if transmission_match:
        transmission = transmission_match.group(1).title()

    engine_match = re.search(
        r"(\d{3,4})\s*to\s*(\d{3,4})\s*cc",
        description,
        re.I
    )

    if engine_match:

        engine = engine_match.group(1)

    else:

        engine_match = re.search(
            r"(\d{3,4})\s*cc",
            description,
            re.I
        )

        if engine_match:
            engine = engine_match.group(1)

    mileage_match = re.search(
        r"mileage of\s*([\d\.]+)",
        description,
        re.I
    )

    if mileage_match:
        mileage = mileage_match.group(1)

    seat_match = re.search(
        r"(\d)\s*seater",
        description,
        re.I
    )

    if seat_match:
        seating = seat_match.group(1)

    rating = np.nan

    if "aggregateRating" in ld_json:

        rating = ld_json["aggregateRating"].get(
            "ratingValue",
            np.nan
        )

    return {

        "Car Name": ld_json.get("name", np.nan),
        "Brand": brand,
        "Price": price,
        "Fuel Type": fuel,
        "Transmission": transmission,
        "Engine (cc)": engine,
        "Mileage": mileage,
        "Seating Capacity": seating,
        "User Rating": rating,
        "URL": url

    }

In [23]:
all_cars = []

for i, url in enumerate(all_car_urls, start=1):

    print(f"{i}/{len(all_car_urls)} : {url}")

    try:

        car_data = scrape_car(url)

        if car_data:
            all_cars.append(car_data)

    except Exception as e:

        print(f"Error : {e}")

    time.sleep(random.uniform(1,2))

1/682 : https://www.carwale.com/aston-martin-cars/db11/
2/682 : https://www.carwale.com/aston-martin-cars/db12/
3/682 : https://www.carwale.com/aston-martin-cars/db9/
4/682 : https://www.carwale.com/aston-martin-cars/dbx/
5/682 : https://www.carwale.com/aston-martin-cars/rapide/
6/682 : https://www.carwale.com/aston-martin-cars/vanquish/
7/682 : https://www.carwale.com/aston-martin-cars/vantage/
8/682 : https://www.carwale.com/aston-martin-cars/virage/
9/682 : https://www.carwale.com/audi-cars/a4/
10/682 : https://www.carwale.com/audi-cars/a5-cabriolet/
11/682 : https://www.carwale.com/audi-cars/a5/
12/682 : https://www.carwale.com/audi-cars/a6/
13/682 : https://www.carwale.com/audi-cars/a8-l/
14/682 : https://www.carwale.com/audi-cars/e-tron-gt/
15/682 : https://www.carwale.com/audi-cars/e-tron-sportback/
16/682 : https://www.carwale.com/audi-cars/e-tron/
17/682 : https://www.carwale.com/audi-cars/new-a3/
18/682 : https://www.carwale.com/audi-cars/new-q3/
19/682 : https://www.carwale.

In [25]:
raw_df = pd.DataFrame(all_cars)

print(raw_df.shape)

raw_df.head()

(514, 10)


,Car Name,Brand,Price,Fuel Type,Transmission,Engine (cc),Mileage,Seating Capacity,User Rating,URL
0,Aston Martin DB11,Aston Martin,31121622.0,NaN,Automatic,5198,8.9,4,4.5,https://www.carwale.com/aston-martin-cars/db11/
1,Aston Martin DB12,Aston Martin,43418919.0,NaN,Automatic,5198,12.75,4,4.1,https://www.carwale.com/aston-martin-cars/db12/
2,Aston Martin DBX,Aston Martin,38200000.0,NaN,Automatic,3982,10.1,5,4.7,https://www.carwale.com/aston-martin-cars/dbx/
3,Aston Martin Vanquish,Aston Martin,83716216.0,NaN,Automatic,5203,NaN,2,4.9,https://www.carwale.com/aston-martin-cars/vanq...
4,Aston Martin Vantage,Aston Martin,37743243.0,NaN,Automatic,3982,NaN,2,5.0,https://www.carwale.com/aston-martin-cars/vant...


In [27]:
print(f"Total URLs Collected : {len(all_car_urls)}")
print(f"Successfully Scraped : {len(all_cars)}")

raw_df.info()

raw_df.isnull().sum()

Total URLs Collected : 682
Successfully Scraped : 514
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514 entries, 0 to 513
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Car Name          514 non-null    object 
 1   Brand             514 non-null    object 
 2   Price             514 non-null    float64
 3   Fuel Type         4 non-null      object 
 4   Transmission      401 non-null    object 
 5   Engine (cc)       320 non-null    object 
 6   Mileage           144 non-null    object 
 7   Seating Capacity  362 non-null    object 
 8   User Rating       393 non-null    float64
 9   URL               514 non-null    object 
dtypes: float64(2), object(8)
memory usage: 40.3+ KB


Car Name              0
Brand                 0
Price                 0
Fuel Type           510
Transmission        113
Engine (cc)         194
Mileage             370
Seating Capacity    152
User Rating         121
URL                   0
dtype: int64

In [31]:
raw_df.to_csv(
    "carwale_raw.csv",
    index=False
)

print("carwale_raw.csv has been saved successfully!")

carwale_raw.csv has been saved successfully!


In [33]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514 entries, 0 to 513
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Car Name          514 non-null    object 
 1   Brand             514 non-null    object 
 2   Price             514 non-null    float64
 3   Fuel Type         4 non-null      object 
 4   Transmission      401 non-null    object 
 5   Engine (cc)       320 non-null    object 
 6   Mileage           144 non-null    object 
 7   Seating Capacity  362 non-null    object 
 8   User Rating       393 non-null    float64
 9   URL               514 non-null    object 
dtypes: float64(2), object(8)
memory usage: 40.3+ KB


In [35]:
raw_df.isnull().sum()

Car Name              0
Brand                 0
Price                 0
Fuel Type           510
Transmission        113
Engine (cc)         194
Mileage             370
Seating Capacity    152
User Rating         121
URL                   0
dtype: int64

In [37]:
raw_df.duplicated().sum()

0

In [39]:
raw_df.head()

,Car Name,Brand,Price,Fuel Type,Transmission,Engine (cc),Mileage,Seating Capacity,User Rating,URL
0,Aston Martin DB11,Aston Martin,31121622.0,NaN,Automatic,5198,8.9,4,4.5,https://www.carwale.com/aston-martin-cars/db11/
1,Aston Martin DB12,Aston Martin,43418919.0,NaN,Automatic,5198,12.75,4,4.1,https://www.carwale.com/aston-martin-cars/db12/
2,Aston Martin DBX,Aston Martin,38200000.0,NaN,Automatic,3982,10.1,5,4.7,https://www.carwale.com/aston-martin-cars/dbx/
3,Aston Martin Vanquish,Aston Martin,83716216.0,NaN,Automatic,5203,NaN,2,4.9,https://www.carwale.com/aston-martin-cars/vanq...
4,Aston Martin Vantage,Aston Martin,37743243.0,NaN,Automatic,3982,NaN,2,5.0,https://www.carwale.com/aston-martin-cars/vant...


In [41]:
fuel_brand_map = {}

print("Collecting model URLs...")

for brand_url in brand_urls:

    brand_slug = brand_url.split("/")[-2].replace("-cars", "").lower()
    brand_name = brand_slug.capitalize()

    fuel_brand_map[brand_name] = []

    try:

        response = requests.get(brand_url, headers=HEADERS)

        if response.status_code != 200:
            continue

        soup = BeautifulSoup(response.text, "lxml")

        seen_urls = set()

        for tag in soup.find_all("a", href=True):

            href = tag.get("href", "").strip()

            full_url = urljoin(BASE_URL, href)

            if f"/{brand_slug}-cars/" not in href.lower():
                continue

            if re.search(r"\d{4}-\d{4}", href):
                continue

            noise_keywords = [
                "compare",
                "upcoming",
                "old-generation",
                "used",
                "history",
                "discontinued",
                "price-in-",
                "images",
                "colors",
                "videos",
                "expert-reviews",
                "reviews",
                "news",
                "features"
            ]

            if any(word in href.lower() for word in noise_keywords):
                continue

            if href.lower().rstrip("/") == f"/{brand_slug}-cars":
                continue

            if full_url not in seen_urls:
                seen_urls.add(full_url)
                fuel_brand_map[brand_name].append(full_url)

        print(f"{brand_name} : {len(fuel_brand_map[brand_name])} models")

        time.sleep(random.uniform(1,2))

    except Exception as e:
        print(e)

Aston-martin : 15 models
Audi : 36 models
Bentley : 8 models
Bmw : 45 models
Byd : 11 models
Citroen : 19 models
Compare : 0 models
Ferrari : 26 models
Force-motors : 6 models
Honda : 30 models
Hyundai : 48 models
Isuzu : 8 models
Jaguar : 13 models
Jeep : 18 models
Kia : 18 models
Lamborghini : 15 models
Land-rover : 15 models
Lexus : 17 models
Lotus : 6 models
Mahindra : 56 models
Maruti-suzuki : 62 models
Maserati : 14 models
Mclaren : 8 models
Mercedes-benz : 75 models
Mg : 31 models
Mini : 16 models
Nissan : 34 models
Porsche : 17 models
Renault : 15 models
Rolls-royce : 13 models
Skoda : 33 models
Tata : 56 models
Tesla : 6 models
Toyota : 59 models
Upcoming : 0 models
Vinfast : 10 models
Volkswagen : 30 models
Volvo : 18 models


In [43]:
def scrape_fuel_specs(car_url, brand_name):

    response = requests.get(car_url, headers=HEADERS)

    if response.status_code != 200:
        return []

    soup = BeautifulSoup(response.text, "lxml")

    title = soup.find("h1")

    car_name = (
        title.text.replace(brand_name, "").strip()
        if title else "Unknown"
    )

    rating = "Not Rated"

    rating_element = soup.find(
        "p",
        class_=re.compile(r"o-iY.*o-jq")
    )

    if rating_element:

        rating = rating_element.text.strip()

        if "/5" not in rating:
            rating += "/5"

    variant_blocks = (
        soup.find_all("div", data_version_id=True)
        or
        soup.find_all(
            "div",
            class_=re.compile(r"sc-jq0284")
        )
    )

    if not variant_blocks:
        variant_blocks = soup.find_all("div")

    records = []

    for block in variant_blocks:

        block_text = block.get_text(" | ")

        if "Rs." not in block_text:
            continue

        if not any(
            fuel in block_text
            for fuel in ["Petrol","Diesel","CNG","Electric"]
        ):
            continue

        price_match = re.search(
            r"Rs\.\s*\d+(?:\.\d+)?\s*(?:Lakh|Crore)",
            block_text
        )

        price = (
            price_match.group(0)
            if price_match
            else "On Request"
        )

        fuel = "Unknown"

        if "Petrol" in block_text:
            fuel = "Petrol"
        elif "Diesel" in block_text:
            fuel = "Diesel"
        elif "CNG" in block_text:
            fuel = "CNG"
        elif "Electric" in block_text:
            fuel = "Electric"

        transmission = "Unknown"

        if "Manual" in block_text:
            transmission = "Manual"

        elif any(
            t in block_text
            for t in [
                "Automatic",
                "AMT",
                "CVT",
                "DCT",
                "Torque Converter"
            ]
        ):
            transmission = "Automatic"

        engine = "Unknown"

        engine_match = re.search(
            r"(\d+\.\d+L(?::\s*Turbo)?|\d+\s*cc|\d+\s*kWh)",
            block_text
        )

        if engine_match:
            engine = engine_match.group(0)

        elif fuel == "Electric":
            engine = "Electric Motor"

        mileage = "N/A"

        mileage_match = re.search(
            r"(\d+(?:\.\d+)?\s*(?:kmpl|km/kg|km/charge))",
            block_text
        )

        if mileage_match:
            mileage = mileage_match.group(0)

        records.append({

            "Car Name": car_name,
            "Brand": brand_name,
            "Price": price,
            "Fuel Type": fuel,
            "Transmission": transmission,
            "Engine": engine,
            "Mileage": mileage,
            "User Rating": rating

        })

    return records

In [45]:
fuel_records = []

for brand_name, urls in fuel_brand_map.items():

    print(f"\n{brand_name}")

    for car_url in urls:

        print(car_url)

        try:

            fuel_records.extend(
                scrape_fuel_specs(
                    car_url,
                    brand_name
                )
            )

        except Exception as e:

            print(e)

        time.sleep(random.uniform(0.3,0.6))


Aston-martin
https://www.carwale.com/aston-martin-cars/db11/#variants
https://www.carwale.com/aston-martin-cars/vanquish/#variants
https://www.carwale.com/aston-martin-cars/vantage/#variants
https://www.carwale.com/aston-martin-cars/dbx/#variants
https://www.carwale.com/aston-martin-cars/db12/#variants
https://www.carwale.com/aston-martin-cars/db11/
https://www.carwale.com/aston-martin-cars/vanquish/
https://www.carwale.com/aston-martin-cars/vantage/
https://www.carwale.com/aston-martin-cars/dbx/
https://www.carwale.com/aston-martin-cars/db12/
https://www.carwale.com/aston-martin-cars/db9/
https://www.carwale.com/aston-martin-cars/dbs20072012/
https://www.carwale.com/aston-martin-cars/rapide/
https://www.carwale.com/aston-martin-cars/virage/
https://www.carwale.com/hi/aston-martin-cars/

Audi
https://www.carwale.com/audi-cars/new-q3/
https://www.carwale.com/audi-cars/new-q5-third-gen/
https://www.carwale.com/audi-cars/q6-e-tron/
https://www.carwale.com/audi-cars/q9/
https://www.carwal

In [47]:
fuel_df = pd.DataFrame(fuel_records)

fuel_df.drop_duplicates(inplace=True)

print(fuel_df.shape)

fuel_df.head()

(2404, 8)


,Car Name,Brand,Price,Fuel Type,Transmission,Engine,Mileage,User Rating
0,Aston Martin DB11,Aston-martin,Rs. 3.11 Crore,Petrol,Automatic,5198 cc,8.9 kmpl,Not Rated
11,Aston Martin DB11,Aston-martin,Rs. 3.11 Crore,Petrol,Automatic,5198 cc,9 kmpl,Not Rated
18,Aston Martin Vanquish,Aston-martin,Rs. 8.37 Crore,Petrol,Automatic,5203 cc,530 km/charge,Not Rated
25,Aston Martin Vanquish,Aston-martin,Rs. 8.37 Crore,Petrol,Automatic,5203 cc,N/A,Not Rated
29,Aston Martin Vanquish,Aston-martin,Rs. 8.37 Crore,Petrol,Automatic,5203 cc,530 km/charge,Not Rated


In [53]:
fuel_df.to_csv(
    "carwale_extracted_data.csv",
    index=False
)

print("carwale_extracted_data.csv saved successfully!")

carwale_extracted_data.csv saved successfully!


In [55]:
fuel_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2404 entries, 0 to 16454
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Car Name      2404 non-null   object
 1   Brand         2404 non-null   object
 2   Price         2404 non-null   object
 3   Fuel Type     2404 non-null   object
 4   Transmission  2404 non-null   object
 5   Engine        2404 non-null   object
 6   Mileage       2404 non-null   object
 7   User Rating   2404 non-null   object
dtypes: object(8)
memory usage: 169.0+ KB


In [57]:
fuel_df.isnull().sum()

Car Name        0
Brand           0
Price           0
Fuel Type       0
Transmission    0
Engine          0
Mileage         0
User Rating     0
dtype: int64

In [59]:
fuel_df.duplicated().sum()

0

In [61]:
fuel_df.head()

,Car Name,Brand,Price,Fuel Type,Transmission,Engine,Mileage,User Rating
0,Aston Martin DB11,Aston-martin,Rs. 3.11 Crore,Petrol,Automatic,5198 cc,8.9 kmpl,Not Rated
11,Aston Martin DB11,Aston-martin,Rs. 3.11 Crore,Petrol,Automatic,5198 cc,9 kmpl,Not Rated
18,Aston Martin Vanquish,Aston-martin,Rs. 8.37 Crore,Petrol,Automatic,5203 cc,530 km/charge,Not Rated
25,Aston Martin Vanquish,Aston-martin,Rs. 8.37 Crore,Petrol,Automatic,5203 cc,N/A,Not Rated
29,Aston Martin Vanquish,Aston-martin,Rs. 8.37 Crore,Petrol,Automatic,5203 cc,530 km/charge,Not Rated


In [63]:
raw_df = pd.read_csv("carwale_raw.csv")
fuel_df = pd.read_csv("carwale_extracted_data.csv")

print("Raw Dataset :", raw_df.shape)
print("Fuel Dataset :", fuel_df.shape)

Raw Dataset : (514, 10)
Fuel Dataset : (2404, 8)


In [65]:
raw_df["Model"] = raw_df.apply(
    lambda row: row["Car Name"].replace(row["Brand"], "").strip()
    if pd.notna(row["Brand"]) and pd.notna(row["Car Name"])
    else np.nan,
    axis=1
)

fuel_df["Model"] = fuel_df.apply(
    lambda row: row["Car Name"].replace(row["Brand"], "").strip()
    if pd.notna(row["Brand"]) and pd.notna(row["Car Name"])
    else np.nan,
    axis=1
)

In [67]:
raw_df["Model"] = (
    raw_df["Model"]
    .str.lower()
    .str.strip()
)

fuel_df["Model"] = (
    fuel_df["Model"]
    .str.lower()
    .str.strip()
)

In [69]:
fuel_df = fuel_df.drop_duplicates(
    subset=["Brand", "Model", "Fuel Type"]
)

print(fuel_df.shape)

(799, 9)


In [71]:
fuel_lookup = (
    fuel_df
    .dropna(subset=["Fuel Type"])
    .drop_duplicates(subset=["Brand", "Model"])
    .set_index(["Brand", "Model"])["Fuel Type"]
    .to_dict()
)

In [73]:
raw_df["Fuel Type"] = raw_df.apply(
    lambda row: fuel_lookup.get(
        (row["Brand"], row["Model"]),
        row["Fuel Type"]
    ),
    axis=1
)

In [75]:
raw_df.drop(columns=["Model"], inplace=True)

In [77]:
final_df = raw_df.copy()

final_df.drop_duplicates(inplace=True)

final_df.reset_index(
    drop=True,
    inplace=True
)

In [79]:
final_df.to_csv(
    "carwale_final.csv",
    index=False
)

print("carwale_final.csv saved successfully!")

carwale_final.csv saved successfully!


In [81]:
print(final_df.shape)

final_df.head()

(514, 10)


,Car Name,Brand,Price,Fuel Type,Transmission,Engine (cc),Mileage,Seating Capacity,User Rating,URL
0,Aston Martin DB11,Aston Martin,31121622.0,NaN,Automatic,5198.0,8.90,4.0,4.5,https://www.carwale.com/aston-martin-cars/db11/
1,Aston Martin DB12,Aston Martin,43418919.0,NaN,Automatic,5198.0,12.75,4.0,4.1,https://www.carwale.com/aston-martin-cars/db12/
2,Aston Martin DBX,Aston Martin,38200000.0,NaN,Automatic,3982.0,10.10,5.0,4.7,https://www.carwale.com/aston-martin-cars/dbx/
3,Aston Martin Vanquish,Aston Martin,83716216.0,NaN,Automatic,5203.0,NaN,2.0,4.9,https://www.carwale.com/aston-martin-cars/vanq...
4,Aston Martin Vantage,Aston Martin,37743243.0,NaN,Automatic,3982.0,NaN,2.0,5.0,https://www.carwale.com/aston-martin-cars/vant...


In [83]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 514 entries, 0 to 513
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Car Name          514 non-null    object 
 1   Brand             514 non-null    object 
 2   Price             514 non-null    float64
 3   Fuel Type         296 non-null    object 
 4   Transmission      401 non-null    object 
 5   Engine (cc)       320 non-null    float64
 6   Mileage           144 non-null    float64
 7   Seating Capacity  362 non-null    float64
 8   User Rating       393 non-null    float64
 9   URL               514 non-null    object 
dtypes: float64(5), object(5)
memory usage: 40.3+ KB


In [85]:
final_df.duplicated().sum()

0

In [87]:
# Fill remaining missing values

# Categorical columns
final_df["Fuel Type"] = final_df["Fuel Type"].fillna("Unknown")
final_df["Transmission"] = final_df["Transmission"].fillna("Unknown")
final_df["User Rating"] = final_df["User Rating"].fillna("Not Rated")

# Numeric columns stored as object
final_df["Engine (cc)"] = (
    final_df["Engine (cc)"]
    .astype(object)
    .fillna("Unknown")
)

final_df["Mileage"] = (
    final_df["Mileage"]
    .astype(object)
    .fillna("Unknown")
)

final_df["Seating Capacity"] = (
    final_df["Seating Capacity"]
    .astype(object)
    .fillna("Unknown")
)

In [89]:
final_df.isnull().sum()

Car Name            0
Brand               0
Price               0
Fuel Type           0
Transmission        0
Engine (cc)         0
Mileage             0
Seating Capacity    0
User Rating         0
URL                 0
dtype: int64

In [91]:
final_df.to_csv(
    "carwale_final.csv",
    index=False
)

print("carwale_final.csv saved successfully!")

carwale_final.csv saved successfully!
